In [ ]:
# ⚙️ Global Config & Services (using centralized modules)

import json
import sys
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

# Add parent directory to path and change to project root
import os

# Get the notebook's current directory and find project root
notebook_dir = Path.cwd()
if notebook_dir.name == "notebooks":
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Change to project root and add to path
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📂 Working directory: {os.getcwd()}")

from src.services.llm_services import (
    load_config,
    get_llm,
    get_text_embeddings,
    validate_api_keys,
    print_config_summary
)

# Load environment variables
load_dotenv()

# Load configuration from config.yaml (now we're in project root)
config = load_config("src/config/config.yaml")

# Validate API keys
validate_api_keys(config, verbose=True)

# Print summary
print_config_summary(config)


📂 Working directory: c:\Users\User\Desktop\Operation Legder-Mind
✅ Config loaded:
  LLM: openrouter (openai/gpt-4o-mini)
  Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2
  Temperature: 0.2
  Artifacts: ./artifacts


c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:375: UserWarning: ⚠️  OPENAI_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:375: UserWarning: ⚠️  GROQ_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")
c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:375: UserWarning: ⚠️  COHERE_API_KEY not found in environment
  warnings.warn(f"⚠️  {key} not found in environment")


In [2]:
# Initialize LLM and Embeddings using factories from llm_services
llm = get_llm(config)
embeddings = get_text_embeddings(config)

print(f"✅ LLM: {config['llm_provider']} / {config['llm_model']}")
print(f"✅ Embeddings: {config['text_emb_provider']} / {config['text_emb_model']}")

# Verify API key with test completion
print("\n🔍 Testing LLM API connection...")
try:
    test_response = llm.invoke("Say 'API working!' if you can read this.")
    test_msg = test_response.content if hasattr(test_response, 'content') else str(test_response)
    print(f"✅ LLM API verified: {test_msg[:50]}")
except Exception as e:
    print(f"❌ LLM API test failed: {e}")
    print("⚠️  Please check your .env file and API key configuration.")


c:\Users\User\Desktop\Operation Legder-Mind\src\services\llm_services.py:129: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ LLM: openrouter / gpt-4o-mini
✅ Embeddings: sbert / sentence-transformers/all-MiniLM-L6-v2

🔍 Testing LLM API connection...
✅ LLM API verified: API working!


**Document Loading**

In [11]:
from langchain_community.document_loaders import (
                                                PyPDFLoader, 
                                                DirectoryLoader, 
                                                TextLoader
                                                )

from langchain_core.documents import Document

pdf_dir = Path(config["data_root"]) / "pdfs"
pdf_dir.mkdir(parents=True, exist_ok=True)


# Try loading PDFs
pdf_files = list(pdf_dir.glob("*.pdf"))
documents=[]
if len(pdf_files) == 0:
    print("⚠️ No PDFs found. Creating sample document...")

    sample_content = """
    Annual Financial Report - 2025

    Company Overview:
    ABC Technologies is a financial technology company
    providing digital payment and banking solutions.

    Revenue:
    During the financial year 2025, the company generated
    total revenue of $5,000,000. Subscription services
    contributed $2,000,000, while transaction fees
    generated $3,000,000.

    Operating Expenses:
    The company reported operating expenses of $3,200,000.
    Employee salaries accounted for $1,500,000.
    Infrastructure costs were $800,000.
    Marketing expenses were $500,000.
    Administrative expenses were $400,000.

    Net Profit:
    The company's net profit for 2025 was $1,800,000.

    Financial Performance:
    Revenue increased by 15% compared to the previous
    financial year. Operating expenses increased by 8%.

    Cash Flow:
    Operating activities generated $2,100,000 in cash.
    Investing activities consumed $600,000.
    Financing activities generated $200,000.

    Risk Management:
    The company identified foreign exchange fluctuations,
    cybersecurity threats, and regulatory changes as
    significant business risks.
    """

    documents.append(
        Document(
            page_content=sample_content,
            metadata={
                "source": "sample_financial_report.txt",
                "page": 1,
                "document_type": "sample"
            }
        )
    )
else:
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
        documents.extend(docs)



print(f"✅ Loaded {len(documents)} document pages")
print(f"  Total characters: {sum(len(d.page_content) for d in documents):,}")



✅ Loaded 142 document pages
  Total characters: 639,429


# Clean Document 

In [13]:
import re

def clean_document(text: str) -> str:

    # Remove page numbers such as Page 25
    text = re.sub(
        r"(?im)^\s*page\s+\d+\s*$",
        "",
        text
    )

    # Remove extra whitespace
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


for doc in documents:
    doc.page_content = clean_document(doc.page_content)

---

#  Chunking 


In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
)

def get_splitter(strategy: str, chunk_size: int = 1500, chunk_overlap: int = 150):
    """
    Return a text splitter based on the specified strategy.
    
    Args:
        strategy: One of "recursive", "fixed", or "sentence"
        chunk_size: Maximum characters per chunk
        chunk_overlap: Overlap between consecutive chunks
        
    Returns:
        A LangChain text splitter instance
    """
    if strategy == "recursive": 
        return RecursiveCharacterTextSplitter(
                                            chunk_size=chunk_size,
                                            chunk_overlap=chunk_overlap,
                                            separators=["\n\n", "\n", ". ", ", ", " ", ""]
                                            )

    elif strategy == "fixed":
        return CharacterTextSplitter(
                                    chunk_size=chunk_size,
                                    chunk_overlap=chunk_overlap,
                                    separator=" "
                                    )

    elif strategy == "sentence":
        pass 

    else:
        raise ValueError("Unknown stratergy")


# Test all strategies
# strategies = ["recursive", "fixed", "sentence"]
strategies = ["recursive"]
split_results = {}

for strategy in strategies:
    splitter = get_splitter(strategy)
    chunks = splitter.split_documents(documents)
    
    # Add strategy to metadata
    for chunk in chunks: 
        chunk.metadata["splitter"] = strategy
    
    split_results[strategy] = chunks
    print(f"✅ {strategy:10s}: {len(chunks):4d} chunks")

if split_results["recursive"]:
    example = split_results["recursive"][0]

    print(f"Length: {len(example.page_content)} chars")
    print(f"Content: {example.page_content[:150]}...")
    print(f"Metadata: {example.metadata}")
else:
    print("No chunks generated.")


✅ recursive :  543 chunks
Length: 586 chars
Content: Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our  
miss...
Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2025-03-20T10:48:37+05:30', 'moddate': '2025-03-22T09:33:29-04:00', 'trapped': '/False', 'source': 'data\\pdfs\\AEE Mini Project 01 2024 Annual Report.pdf', 'total_pages': 142, 'page': 1, 'page_label': 'b', 'splitter': 'recursive'}
